In [ ]:
# 1. Install necessary libraries
!pip install -q -U google-genai pymupdf

# 2. Imports
import os
import json
import fitz  # PyMuPDF
from google import genai
from google.genai import types
from google.colab import userdata, drive

drive.mount('/content/drive')

BASE_PATH = "path to your file"
RAW_PDF_DIR = f"{BASE_PATH}/raw_pdfs"
INDEX_DIR = f"{BASE_PATH}/indices"
CONTENT_DIR = f"{BASE_PATH}/content"

for path in [RAW_PDF_DIR, INDEX_DIR, CONTENT_DIR]:
    os.makedirs(path, exist_ok=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 8.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.1 which is incompatible.
Mounted at /content/drive


In [ ]:
import os
import json
import pandas as pd
import time
import re
from google import genai

# --- CONFIGURATION ---
api_key = 'API_KEY'
client = genai.Client(api_key=api_key)

BASE_PATH = "/content/drive/MyDrive/L-PINE_PageIndex"
RAW_PDF_DIR = f"{BASE_PATH}/raw_pdfs"
INDEX_DIR = f"{BASE_PATH}/indices"
CONTENT_DIR = f"{BASE_PATH}/content"
CSV_FILE = "juris_mind_test_suite.csv"

def get_full_text(doc_id):
    """Reads and concatenates all page markdown files for a doc."""
    doc_path = os.path.join(CONTENT_DIR, doc_id)
    if not os.path.exists(doc_path):
        return None

    try:
        files = sorted(os.listdir(doc_path), key=lambda x: int(re.findall(r'\d+', x)[0]))
    except Exception:
        files = sorted(os.listdir(doc_path))

    full_text = ""
    for file in files:
        if file.endswith(".md"):
            with open(os.path.join(doc_path, file), "r", encoding="utf-8") as f:
                full_text += f"\n--- {file.upper()} ---\n{f.read()}\n"
    return full_text

def generate_doc_dataset(doc_id):
    """Generates 20 high-fidelity Q-A pairs with a mix of direct and inferential questions."""

    # 1. Resume Logic
    if os.path.exists(CSV_FILE):
        df_existing = pd.read_csv(CSV_FILE)
        existing_count = len(df_existing[df_existing['Doc_ID'] == doc_id])
        if existing_count >= 20:
            print(f"Doc '{doc_id}' already has {existing_count} questions. Skipping.")
            return
    else:
        df_existing = pd.DataFrame(columns=['Question', 'Ground_Truth_Answer', 'Doc_ID', 'Page_Numbers', 'Category', 'Type'])

    # 2. Load Index and Raw Text
    idx_path = os.path.join(INDEX_DIR, f"{doc_id}_index.json")
    raw_text = get_full_text(doc_id)

    if not os.path.exists(idx_path) or not raw_text:
        print(f"Data missing for {doc_id}. Check INDEX_DIR and CONTENT_DIR.")
        return

    with open(idx_path, 'r', encoding="utf-8") as f:
        doc_index = json.load(f)

    index_summary = "\n".join([f"Page {k}: {v}" for k, v in doc_index.items()])

    # 3. The Prompt: Mixed Question Types & Title-Neutral
    prompt = f"""
    You are a Senior Legal Bar Examiner. Create 20 easy-to-moderate test questions for the document: {doc_id}.

    CONTEXT (INDEX):
    {index_summary}

    CONTEXT (RAW TEXT):
    {raw_text[:900000]}

    TASK CONSTRAINTS:
    1. ZERO AMBIGUITY BUT NO SPOILERS: Do NOT use the formal title of the document or filename.
       Use domain-specific phrases like 'In industrial design registration...' or 'Regarding invention filings...'.

    2. QUESTION MIX (TOTAL 20):
       - 14-15 DIRECT QUESTIONS: Fact-based (e.g., specific fees, form numbers, or fixed deadlines).
       - 5-6 INFERENTIAL/SUBJECTIVE QUESTIONS: These should not be direct "lookup" answers.
         Ask about procedural consequences, the rationale for a classification, or 'what happens if'
         scenarios based on the rules. (e.g., 'What are the implications if a proprietor fails to
         comply with marking requirements?').

    3. FACTUAL GROUND TRUTH: Even for subjective questions, the answer must be strictly derived from
       the RAW TEXT. Do not hallucinate outside legal knowledge.

    4. DISTRIBUTION: Spread questions across the entire document length.

    OUTPUT FORMAT:
    Return a JSON list of objects ONLY. Include a "Type" key ('Direct' or 'Inferential').
    [
      {{
        "Question": "...",
        "Ground_Truth_Answer": "...",
        "Page_Numbers": "...",
        "Category": "...",
        "Type": "Direct"
      }},
      ...
    ]
    """

    print(f"Generating 20 Mixed-Type questions for: {doc_id}...")

    try:
        response = client.models.generate_content(model="gemini-flash-latest", contents=prompt)

        # Clean JSON response
        raw_json = response.text.replace("```json", "").replace("```", "").strip()
        new_data = json.loads(raw_json)

        new_rows = []
        for item in new_data:
            new_rows.append({
                'Question': item['Question'],
                'Ground_Truth_Answer': item['Ground_Truth_Answer'],
                'Doc_ID': doc_id,
                'Page_Numbers': item['Page_Numbers'],
                'Category': item['Category'],
                'Type': item.get('Type', 'Direct')
            })

        df_new = pd.DataFrame(new_rows)
        df_final = pd.concat([df_existing, df_new], ignore_index=True)
        df_final.to_csv(CSV_FILE, index=False)
        print(f"Successfully added 20 questions (mixed types) for {doc_id}.")

    except Exception as e:
        print(f" Error for {doc_id}: {e}")

In [ ]:
# --- EXECUTION ---
# generate_doc_dataset("indian_patent_rules_2003__1_till2021")
# generate_doc_dataset("design-rules-2001")
# generate_doc_dataset("trademark_rules")
# generate_doc_dataset("trademark_rules-75-93")
generate_doc_dataset("tables_patentrules")